# 08  -  Explainability: SHAP & LIME

**Inputs:** `models/`, `outputs/*_features.csv`  
**Outputs:** SHAP plots, `outputs/shap_feature_importance.csv`

In [ ]:
exec(open('00_config.py').read())

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib, warnings
warnings.filterwarnings('ignore')

import shap
from lime import lime_tabular
from scipy.stats import spearmanr

cc = pd.read_csv(CC_FEATURES_CSV)
mm = pd.read_csv(MM_FEATURES_CSV)
hi = pd.read_csv(HI_FEATURES_CSV)
print('Loaded')


## 1. SHAP  -  Global Explanations

In [ ]:
def remove_hi_leaks(df):
    leak_kw = ['dev_','overbill','billing_variab','drg_fraud_rate','max_deviation',
               'multi_field','is_overbill','amount_deviation','amount_zscore',
               'is_large_trans','is_micro_trans','is_high_value','is_service_dom']
    exact   = ['Total','Service','Medical','Approved_Cost','service_to_total_ratio','NHIS ID No','Folder No','No.']
    drop    = [c for c in df.columns if any(k in c for k in leak_kw) or c in exact]
    return df.drop(columns=drop, errors='ignore')

from sklearn.ensemble import RandomForestClassifier
shap_results = {}

for domain, df_raw, key in [
    ('Credit Card',      cc, 'rf_credit_card'),
    ('Mobile Money',     mm, 'rf_mobile_money'),
    ('Health Insurance', hi, 'rf_health_insurance'),
]:
    print(f'\n--- SHAP: {domain} ---')
    df = remove_hi_leaks(df_raw.copy()) if domain == 'Health Insurance' else df_raw
    X  = df.drop(columns=[TARGET_COL]).select_dtypes(include=np.number).fillna(0)
    y  = df_raw[TARGET_COL]

    model_path = os.path.join(MODEL_DIR, f'{key}.pkl')
    if os.path.exists(model_path):
        model = joblib.load(model_path)
    else:
        print('  Model not found  -  fitting fresh RF')
        model = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                       random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(X, y)

    explainer = shap.TreeExplainer(model)
    X_sample  = X.sample(min(500, len(X)), random_state=RANDOM_STATE)
    sv_raw    = explainer.shap_values(X_sample)

    # For binary RF, shap_values is a list [class0, class1]; take class1 (fraud)
    sv = sv_raw[1] if isinstance(sv_raw, list) else (sv_raw[:,:,1] if sv_raw.ndim == 3 else sv_raw)

    shap_results[domain] = {'explainer': explainer, 'shap_values': sv,
                             'X_sample': X_sample, 'feature_names': X.columns.tolist()}

    mean_abs = pd.Series(np.abs(sv).mean(axis=0), index=X.columns)
    print('  Top 10:')
    for feat, val in mean_abs.nlargest(10).items():
        print(f'    {feat:<35} {val:.4f}')

print('\nSHAP done')

## 2. SHAP Summary Plots

In [ ]:
os.makedirs(FIGURE_DIR, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, (domain, res) in zip(axes, shap_results.items()):
    sv, feats = res['shap_values'], res['feature_names']
    mean_abs  = np.abs(sv).mean(axis=0)
    top_idx   = np.argsort(mean_abs)[-15:]
    ax.barh([feats[i] for i in top_idx], mean_abs[top_idx], color='#E53935', alpha=0.8)
    ax.set_title(f'{domain}\nTop 15 SHAP Features')
    ax.set_xlabel('Mean |SHAP|')

plt.suptitle('Global Feature Importance (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, '08_shap_global.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-domain: summary beeswarm + force plot + dependence plot
for domain, res in shap_results.items():
    sv, X_s, feats = res['shap_values'], res['X_sample'], res['feature_names']
    df_plot = pd.DataFrame(X_s.values, columns=feats)
    safe    = domain.lower().replace(' ', '_')
    exp     = res['explainer']

    # Summary beeswarm
    fig1, _ = plt.subplots(figsize=(10, 6))
    shap.summary_plot(sv, df_plot, feature_names=feats, max_display=15, show=False, plot_type='dot')
    plt.title(f'{domain}  -  SHAP Beeswarm', fontweight='bold')
    plt.tight_layout()
    fig1.savefig(os.path.join(FIGURE_DIR, f'08_shap_summary_{safe}.png'), dpi=150, bbox_inches='tight')
    plt.close(fig1)

    # Force plot for a fraud instance
    df_raw = {'Credit Card': cc, 'Mobile Money': mm, 'Health Insurance': hi}[domain]
    fraud_rows = df_raw[df_raw[TARGET_COL] == 1]
    if len(fraud_rows) > 0:
        instance = (fraud_rows.drop(columns=[TARGET_COL], errors='ignore')
                    .select_dtypes(include=np.number).fillna(0)
                    .reindex(columns=feats, fill_value=0).iloc[:1])
        sv_i = exp.shap_values(instance)
        sv_i = sv_i[1] if isinstance(sv_i, list) else sv_i
        sv_flat = sv_i.flatten()[:len(feats)]
        top_idx = np.argsort(np.abs(sv_flat))[-10:][::-1]
        colors  = ['#E53935' if v > 0 else '#1565C0' for v in sv_flat[top_idx]]
        fig2, ax2 = plt.subplots(figsize=(9, 5))
        ax2.barh([feats[int(i)] for i in top_idx], sv_flat[top_idx], color=colors, alpha=0.85)
        ax2.axvline(0, color='black', linewidth=0.8)
        ax2.set_title(f'{domain}  -  SHAP Force Plot (fraud instance)', fontweight='bold')
        ax2.set_xlabel('SHAP Value  (red=fraud, blue=legit)')
        plt.tight_layout()
        fig2.savefig(os.path.join(FIGURE_DIR, f'08_shap_force_{safe}.png'), dpi=150, bbox_inches='tight')
        plt.close(fig2)

    # Dependence plot  -  top feature
    top_feat = feats[int(np.argmax(np.abs(sv).mean(axis=0)))]
    fig3, ax3 = plt.subplots(figsize=(8, 5))
    ax3.scatter(df_plot[top_feat], sv[:, feats.index(top_feat)], alpha=0.3, s=10, c='#E53935')
    ax3.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax3.set_xlabel(top_feat); ax3.set_ylabel(f'SHAP({top_feat})')
    ax3.set_title(f'{domain}  -  Dependence: {top_feat}', fontweight='bold')
    plt.tight_layout()
    fig3.savefig(os.path.join(FIGURE_DIR, f'08_shap_dependence_{safe}.png'), dpi=150, bbox_inches='tight')
    plt.close(fig3)
    print(f'  [{domain}] plots saved')

print('\nAll SHAP plots saved')

## 3. LIME

In [ ]:
lime_results = {}
for domain, df_raw, key in [
    ('Credit Card',      cc, 'rf_credit_card'),
    ('Mobile Money',     mm, 'rf_mobile_money'),
    ('Health Insurance', hi, 'rf_health_insurance'),
]:
    print(f'\n--- LIME: {domain} ---')
    df = remove_hi_leaks(df_raw.copy()) if domain == 'Health Insurance' else df_raw
    X  = df.drop(columns=[TARGET_COL], errors='ignore').select_dtypes(include=np.number).fillna(0)

    model_path = os.path.join(MODEL_DIR, f'{key}.pkl')
    if os.path.exists(model_path):
        model = joblib.load(model_path)
    else:
        from sklearn.ensemble import RandomForestClassifier
        model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
        model.fit(X, df_raw[TARGET_COL])

    lime_exp = lime_tabular.LimeTabularExplainer(
        X.values, feature_names=X.columns.tolist(),
        class_names=['Legitimate','Fraud'], mode='classification', random_state=RANDOM_STATE
    )

    fraud_rows = df_raw[df_raw[TARGET_COL] == 1]
    instance = (fraud_rows if len(fraud_rows) > 0 else df_raw).drop(columns=[TARGET_COL], errors='ignore')               .select_dtypes(include=np.number).fillna(0).reindex(columns=X.columns, fill_value=0).iloc[0].values

    expl = lime_exp.explain_instance(instance, model.predict_proba, num_features=10, num_samples=500)
    lime_results[domain] = expl

    print('  Top 5:')
    for feat, w in expl.as_list()[:5]:
        print(f'    {feat[:50]:<52} {w:+.4f}  {"-> Fraud" if w > 0 else "-> Legit"}')

print('\nLIME done')

## 4. Cross-Domain SHAP Correlation

In [ ]:
importance = {d: pd.Series(np.abs(r['shap_values']).mean(axis=0), index=r['feature_names'])
               for d, r in shap_results.items()}

imp_df = pd.DataFrame(importance).fillna(0)
imp_df.to_csv(os.path.join(OUTPUT_DIR, 'shap_feature_importance.csv'))
print(f'Saved -> {OUTPUT_DIR}/shap_feature_importance.csv')

print('\nSpearman rank correlation of SHAP importance:')
domains = list(importance.keys())
for i in range(len(domains)):
    for j in range(i+1, len(domains)):
        d1, d2 = domains[i], domains[j]
        common = list(set(importance[d1].index) & set(importance[d2].index))
        if len(common) > 2:
            r, p = spearmanr(importance[d1][common].values, importance[d2][common].values)
            print(f'  {d1[:3]} <-> {d2[:3]}: r={r:.4f}  p={p:.4f}')
        else:
            print(f'  {d1[:3]} <-> {d2[:3]}: no common features')

## 5. Explainability Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 10))

for idx, (domain, res) in enumerate(shap_results.items()):
    ax      = fig.add_subplot(2, 3, idx+1)
    sv, feats = res['shap_values'], res['feature_names']
    mean_abs  = np.abs(sv).mean(axis=0)
    top_idx   = np.argsort(mean_abs)[-8:]
    ax.barh([feats[i] for i in top_idx], mean_abs[top_idx], color='#E53935', alpha=0.8)
    ax.set_title(f'{domain}\nSHAP Top 8')
    ax.set_xlabel('Mean |SHAP|')

ax4     = fig.add_subplot(2, 3, (4, 6))
top_rows = imp_df.sum(axis=1).nlargest(15).index
imp_sub  = imp_df.loc[top_rows]
im = ax4.imshow(imp_sub.values, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax4, label='Mean |SHAP|')
ax4.set_xticks(range(len(imp_sub.columns))); ax4.set_xticklabels(imp_sub.columns, fontsize=10)
ax4.set_yticks(range(len(imp_sub.index)));   ax4.set_yticklabels(imp_sub.index, fontsize=8)
ax4.set_title('Cross-Domain Feature Importance Heatmap')

plt.suptitle('UniFraud-GH Explainability Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, '08_explainability_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved')